In [1]:
# Import libraries and other configurations
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

sns.set_theme(style="whitegrid")

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 121

In [2]:
# Load the cleaned dataset
df = pd.read_csv(
    "../data/processed/cleaned_donor_data.csv",
    dtype={"donor_postal_code": "str"}
)

df.columns.tolist()

['donor_unique_id',
 'donor_postal_code',
 'donor_age',
 'gender_identity',
 'is_member_flag',
 'is_alumnus_flag',
 'is_parent_flag',
 'has_involvement_flag',
 'preferred_address_type',
 'has_email_flag',
 'consecutive_donor_years',
 'last_fiscal_year_donation',
 'donation_2_fiscal_years_ago',
 'donation_3_fiscal_years_ago',
 'donation_4_fiscal_years_ago',
 'donation_5_fiscal_years_ago',
 'current_fiscal_year_donation',
 'cumulative_donation_amount',
 'donor_indicator_flag']

In [3]:
# Verify dataset loaded correctly
print(f"Dataset Shape: {df.shape}")
df.sample(10, random_state=RANDOM_STATE)

Dataset Shape: (34403, 19)


,donor_unique_id,donor_postal_code,donor_age,gender_identity,is_member_flag,is_alumnus_flag,is_parent_flag,has_involvement_flag,preferred_address_type,has_email_flag,consecutive_donor_years,last_fiscal_year_donation,donation_2_fiscal_years_ago,donation_3_fiscal_years_ago,donation_4_fiscal_years_ago,donation_5_fiscal_years_ago,current_fiscal_year_donation,cumulative_donation_amount,donor_indicator_flag
33267,33371,42301.0,46,Female,0,0,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
26595,26683,45620.0,42,Female,0,0,0,0,Home,0,1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
16653,16704,33427.0,28,Female,0,0,0,0,Home,0,0,0.00,120.00,0.00,0.00,0.00,0.00,120.00,1
21428,21498,54131.0,33,Female,0,0,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1
16873,16924,90265.0,33,Male,0,1,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
19919,19982,54459.0,32,Female,0,0,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
15478,15527,64686.0,42,Male,0,0,0,0,Home,0,8,0.00,0.00,10.00,0.00,0.00,0.00,"9,680.00",1
591,596,12067.0,42,Female,0,0,1,0,Home,0,0,0.00,100.00,1.00,0.00,0.00,0.00,101.00,1
1474,1483,90265.0,42,Female,0,1,0,0,Home,1,0,0.00,0.00,0.00,0.00,0.00,0.00,479.00,1
22846,22922,45856.0,36,Female,0,0,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0


In [4]:
# Verify dataset data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34403 entries, 0 to 34402
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   donor_unique_id               34403 non-null  int64  
 1   donor_postal_code             34312 non-null  str    
 2   donor_age                     34403 non-null  int64  
 3   gender_identity               33912 non-null  str    
 4   is_member_flag                34403 non-null  int64  
 5   is_alumnus_flag               34403 non-null  int64  
 6   is_parent_flag                34403 non-null  int64  
 7   has_involvement_flag          34403 non-null  int64  
 8   preferred_address_type        30370 non-null  str    
 9   has_email_flag                34403 non-null  int64  
 10  consecutive_donor_years       34403 non-null  int64  
 11  last_fiscal_year_donation     34403 non-null  float64
 12  donation_2_fiscal_years_ago   34403 non-null  float64
 13  donation_3_f

In [5]:
# Data types summary
dtype_summary = (
    df.dtypes
      .astype(str)
      .value_counts()
      .reset_index()
      .style.hide(axis="index")
)

dtype_summary.columns = ["Data Type", "Count"]
dtype_summary

index,count
int64,9
float64,7
str,3


In [6]:
# Verify dataset missing values
missing_summary = (
    df.isna()
      .sum()
      .to_frame("Missing Count")
)

missing_summary["Missing %"] = (
    missing_summary["Missing Count"] / len(df) * 100
)

missing_summary = (
    missing_summary
      .sort_values("Missing Count", ascending=False)
)

missing_summary

,Missing Count,Missing %
preferred_address_type,4033,11.72
gender_identity,491,1.43
donor_postal_code,91,0.26
donor_unique_id,0,0.00
last_fiscal_year_donation,0,0.00
cumulative_donation_amount,0,0.00
current_fiscal_year_donation,0,0.00
donation_5_fiscal_years_ago,0,0.00
donation_4_fiscal_years_ago,0,0.00
donation_3_fiscal_years_ago,0,0.00


## Dataset Overview
The finalized cleaned analytical dataset was successfully loaded and verified for feature engineering. The dataset contains 34,403 donor records and 19 features, consistent with the dataset used throughout Phase 2. Data types and missing values match expectations from the data cleaning process, and all required libraries were successfully imported. The only change made was explicitly defining `donor_postal_code` as a string. This was made because it was interpreting postal codes as a float and not a string.

In [7]:
# Define target-related column names
TARGET_SOURCE_COLUMN = "current_fiscal_year_donation"
TARGET_COLUMN = "target_current_fiscal_year_donor_flag"

# Validate the target source before creating the binary target
assert pd.api.types.is_numeric_dtype(df[TARGET_SOURCE_COLUMN]), (
    f"{TARGET_SOURCE_COLUMN} must contain numeric values."
)

# Check for missing values
missing_target_values = df[TARGET_SOURCE_COLUMN].isna().sum()
print(f"Missing target-source values: {missing_target_values:,}")

assert missing_target_values == 0, (
    f"{TARGET_SOURCE_COLUMN} contains "
    f"{missing_target_values:,} missing values."
)

# Check for negative numbers
negative_target_values = (df[TARGET_SOURCE_COLUMN] < 0).sum()
print(f"Negative target-source values: {negative_target_values:,}")

assert negative_target_values == 0, (
    f"{TARGET_SOURCE_COLUMN} contains "
    f"{negative_target_values:,} negative values."
)

Missing target-source values: 0
Negative target-source values: 0


In [8]:
# Create the binary prediction target
df[TARGET_COLUMN] = (
    df[TARGET_SOURCE_COLUMN] > 0
).astype("int8")

print(f"Target column created: {TARGET_COLUMN}")

Target column created: target_current_fiscal_year_donor_flag


In [9]:
# Summarize the target distribution
target_summary = (
    df[TARGET_COLUMN]
    .value_counts()
    .sort_index()
    .rename_axis(TARGET_COLUMN)
    .reset_index(name="count")
)

target_summary["percentage"] = (
    target_summary["count"] / len(df) * 100
).round(2)

target_summary["target_label"] = target_summary[TARGET_COLUMN].map(
    {
        0: "Did not donate",
        1: "Donated",
    }
)

target_summary = target_summary[
    [
        TARGET_COLUMN,
        "target_label",
        "count",
        "percentage",
    ]
]

display(target_summary.style.hide(axis="index").format({"percentage": "{:.2f}%", "count": "{:,}"}))

target_current_fiscal_year_donor_flag,target_label,count,percentage
0,Did not donate,"32,499",94.47%
1,Donated,"1,904",5.53%


In [10]:
# Confirm that the target contains only binary values and matches its source column
expected_target_values = {0, 1}
actual_target_values = set(df[TARGET_COLUMN].unique())

assert actual_target_values == expected_target_values, (
    f"Unexpected target values found: {actual_target_values}"
)

# Confirm that the target matches its source column
target_mismatch_count = (
    df[TARGET_COLUMN]
    != (df[TARGET_SOURCE_COLUMN] > 0).astype("int8")
).sum()

assert target_mismatch_count == 0, (
    f"Found {target_mismatch_count:,} target-definition mismatches."
)

print("Target validation passed.")

Target validation passed.


## Prediction Target and Time Window

The classification target is `target_current_fiscal_year_donor_flag`, which identifies whether an individual donated during the current fiscal year.

The target is derived from `current_fiscal_year_donation`:

- `0`: The individual donated $0 during the current fiscal year
  
- `1`: The individual donated more than $0 during the current fiscal year

The target distribution shows that 1,904 individuals, or 5.53% of the dataset, donated during the current fiscal year. The remaining 32,499 individuals, or 94.47%, did not donate. This indicates a substantial class imbalance that will need to be considered during model evaluation and training.

The intended prediction setup uses the five completed fiscal years before the current fiscal year as the historical observation window. The current fiscal year serves as the outcome period.

The model will therefore use information available before the beginning of the current fiscal year to predict whether an individual donates during that fiscal year.

The original `current_fiscal_year_donation` column will remain in the working dataset temporarily for validation, but it must not be included in the predictor matrix because it directly determines the target.

Until additional documentation is available:

* `cumulative_donation_amount` will not be used directly because it may include current-fiscal-year donations
* `donor_indicator_flag` will not be used as a predictor until its calculation timing and logic are confirmed
* `consecutive_donor_years` will not be used until it is confirmed that it excludes current-fiscal-year activity
* Demographic and engagement fields will be treated as provisionally available
* The exact fiscal-year dates and dataset extraction date should be confirmed before the final model is deployed

The timing and leakage status of all source variables will be evaluated during the formal leakage audit.


In [11]:
# Define leakage audit classifications
VALID_LEAKAGE_CLASSIFICATIONS = {
    "Safe predictor",
    "Potential leakage",
    "Identifier",
    "Target",
    "Excluded feature",
}

# Document the leakage decision for every column
leakage_audit = pd.DataFrame(
    [
        {
            "column": "donor_unique_id",
            "classification": "Identifier",
            "model_treatment": "Exclude from predictors",
            "reason": (
                "Unique record identifier with no meaningful predictive value."
            ),
            "required_action": (
                "Retain only for record tracking and exported predictions."
            ),
        },
        {
            "column": "donor_postal_code",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Geographic information should be available before the "
                "prediction period and does not directly reveal the target."
            ),
            "required_action": (
                "Evaluate cardinality, privacy, and fairness before modeling."
            ),
        },
        {
            "column": "donor_age",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Demographic information does not directly reveal whether "
                "the individual donated during the target period."
            ),
            "required_action": (
                "Document that the exact age reference date is unavailable."
            ),
        },
        {
            "column": "gender_identity",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Demographic information does not directly reveal the "
                "current-fiscal-year donation outcome."
            ),
            "required_action": (
                "Consolidate unknown and missing categories before modeling."
            ),
        },
        {
            "column": "is_member_flag",
            "classification": "Excluded feature",
            "model_treatment": "Exclude from predictors",
            "reason": (
                "The field is constant in the cleaned dataset and therefore "
                "provides no predictive variation."
            ),
            "required_action": "Exclude from the modeling feature set.",
        },
        {
            "column": "is_alumnus_flag",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Alumni status is a relatively stable relationship attribute "
                "and does not directly reveal the target."
            ),
            "required_action": (
                "Retain unless later documentation identifies a timing issue."
            ),
        },
        {
            "column": "is_parent_flag",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Parent status is a relatively stable relationship attribute "
                "and does not directly reveal the target."
            ),
            "required_action": (
                "Retain unless later documentation identifies a timing issue."
            ),
        },
        {
            "column": "has_involvement_flag",
            "classification": "Potential leakage",
            "model_treatment": "Exclude pending verification",
            "reason": (
                "The snapshot date is unknown, so the field may include "
                "involvement established during the target period."
            ),
            "required_action": (
                "Confirm that the value was available at the prediction cutoff."
            ),
        },
        {
            "column": "preferred_address_type",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Contact-preference information does not directly reveal "
                "current-fiscal-year donation behavior."
            ),
            "required_action": (
                "Standardize categories and handle missing values before modeling."
            ),
        },
        {
            "column": "has_email_flag",
            "classification": "Potential leakage",
            "model_treatment": "Exclude pending verification",
            "reason": (
                "The field may reflect contact information added or updated "
                "during the target period."
            ),
            "required_action": (
                "Confirm that the value was available at the prediction cutoff."
            ),
        },
        {
            "column": "consecutive_donor_years",
            "classification": "Potential leakage",
            "model_treatment": "Exclude pending verification",
            "reason": (
                "The count may include donation activity from the current "
                "fiscal year."
            ),
            "required_action": (
                "Confirm the calculation period or reconstruct it from "
                "historical donation columns."
            ),
        },
        {
            "column": "last_fiscal_year_donation",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Represents donation behavior from the completed fiscal year "
                "before the target period."
            ),
            "required_action": "Retain as a historical predictor.",
        },
        {
            "column": "donation_2_fiscal_years_ago",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Represents donation behavior from before the target period."
            ),
            "required_action": "Retain as a historical predictor.",
        },
        {
            "column": "donation_3_fiscal_years_ago",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Represents donation behavior from before the target period."
            ),
            "required_action": "Retain as a historical predictor.",
        },
        {
            "column": "donation_4_fiscal_years_ago",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Represents donation behavior from before the target period."
            ),
            "required_action": "Retain as a historical predictor.",
        },
        {
            "column": "donation_5_fiscal_years_ago",
            "classification": "Safe predictor",
            "model_treatment": "Candidate predictor",
            "reason": (
                "Represents donation behavior from before the target period."
            ),
            "required_action": "Retain as a historical predictor.",
        },
        {
            "column": "current_fiscal_year_donation",
            "classification": "Excluded feature",
            "model_treatment": "Exclude from predictors",
            "reason": (
                "Directly determines the binary modeling target and would "
                "cause target leakage if used as a predictor."
            ),
            "required_action": (
                "Retain only for target construction and validation."
            ),
        },
        {
            "column": "cumulative_donation_amount",
            "classification": "Excluded feature",
            "model_treatment": "Exclude or reconstruct",
            "reason": (
                "The cumulative value may include the current-fiscal-year "
                "donation and therefore reveal target-period information."
            ),
            "required_action": (
                "Replace with a historical value calculated only from "
                "pre-cutoff donation data."
            ),
        },
        {
            "column": "donor_indicator_flag",
            "classification": "Potential leakage",
            "model_treatment": "Exclude pending verification",
            "reason": (
                "The calculation logic and timing are unknown and may include "
                "current-fiscal-year donation behavior."
            ),
            "required_action": (
                "Confirm the source logic and reference date before use."
            ),
        },
        {
            "column": "target_current_fiscal_year_donor_flag",
            "classification": "Target",
            "model_treatment": "Use as modeling target",
            "reason": (
                "Binary outcome created from whether "
                "current_fiscal_year_donation is greater than zero."
            ),
            "required_action": (
                "Use as the target variable and never include it among predictors."
            ),
        },
    ]
)

display(leakage_audit.style.hide(axis="index"))

column,classification,model_treatment,reason,required_action
donor_unique_id,Identifier,Exclude from predictors,Unique record identifier with no meaningful predictive value.,Retain only for record tracking and exported predictions.
donor_postal_code,Safe predictor,Candidate predictor,Geographic information should be available before the prediction period and does not directly reveal the target.,"Evaluate cardinality, privacy, and fairness before modeling."
donor_age,Safe predictor,Candidate predictor,Demographic information does not directly reveal whether the individual donated during the target period.,Document that the exact age reference date is unavailable.
gender_identity,Safe predictor,Candidate predictor,Demographic information does not directly reveal the current-fiscal-year donation outcome.,Consolidate unknown and missing categories before modeling.
is_member_flag,Excluded feature,Exclude from predictors,The field is constant in the cleaned dataset and therefore provides no predictive variation.,Exclude from the modeling feature set.
is_alumnus_flag,Safe predictor,Candidate predictor,Alumni status is a relatively stable relationship attribute and does not directly reveal the target.,Retain unless later documentation identifies a timing issue.
is_parent_flag,Safe predictor,Candidate predictor,Parent status is a relatively stable relationship attribute and does not directly reveal the target.,Retain unless later documentation identifies a timing issue.
has_involvement_flag,Potential leakage,Exclude pending verification,"The snapshot date is unknown, so the field may include involvement established during the target period.",Confirm that the value was available at the prediction cutoff.
preferred_address_type,Safe predictor,Candidate predictor,Contact-preference information does not directly reveal current-fiscal-year donation behavior.,Standardize categories and handle missing values before modeling.
has_email_flag,Potential leakage,Exclude pending verification,The field may reflect contact information added or updated during the target period.,Confirm that the value was available at the prediction cutoff.


In [12]:
# Confirm that each column appears exactly once
duplicate_audit_columns = leakage_audit["column"].duplicated().sum()

dataset_columns = set(df.columns)
audited_columns = set(leakage_audit["column"])

missing_audit_columns = sorted(dataset_columns - audited_columns)
unexpected_audit_columns = sorted(audited_columns - dataset_columns)

invalid_classifications = sorted(
    set(leakage_audit["classification"])
    - VALID_LEAKAGE_CLASSIFICATIONS
)

assert duplicate_audit_columns == 0, (
    f"The leakage audit contains "
    f"{duplicate_audit_columns} duplicate column entries."
)

assert not missing_audit_columns, (
    f"Columns missing from the leakage audit: "
    f"{missing_audit_columns}"
)

assert not unexpected_audit_columns, (
    f"Audit columns not found in the dataset: "
    f"{unexpected_audit_columns}"
)

assert not invalid_classifications, (
    f"Invalid classifications found: "
    f"{invalid_classifications}"
)

print("Leakage audit validation passed.")
print(f"Dataset columns audited: {len(leakage_audit)}")

Leakage audit validation passed.
Dataset columns audited: 20


In [13]:
# Summarize the num of columns in each classification
leakage_audit_summary = (
    leakage_audit["classification"]
    .value_counts()
    .rename_axis("classification")
    .reset_index(name="column_count")
)

display(leakage_audit_summary.style.hide(axis="index"))

classification,column_count
Safe predictor,11
Potential leakage,4
Excluded feature,3
Identifier,1
Target,1


In [14]:
# Reusable column lists for feature engineering and modeling
SAFE_PREDICTOR_COLUMNS = leakage_audit.loc[
    leakage_audit["classification"] == "Safe predictor",
    "column",
].tolist()

POTENTIAL_LEAKAGE_COLUMNS = leakage_audit.loc[
    leakage_audit["classification"] == "Potential leakage",
    "column",
].tolist()

IDENTIFIER_COLUMNS = leakage_audit.loc[
    leakage_audit["classification"] == "Identifier",
    "column",
].tolist()

EXCLUDED_FEATURE_COLUMNS = leakage_audit.loc[
    leakage_audit["classification"] == "Excluded feature",
    "column",
].tolist()

TARGET_COLUMNS = leakage_audit.loc[
    leakage_audit["classification"] == "Target",
    "column",
].tolist()

print("Safe predictors:")
print(SAFE_PREDICTOR_COLUMNS)

print("\nPotential leakage columns:")
print(POTENTIAL_LEAKAGE_COLUMNS)

print("\nIdentifier columns:")
print(IDENTIFIER_COLUMNS)

print("\nExcluded features:")
print(EXCLUDED_FEATURE_COLUMNS)

print("\nTarget-related columns:")
print(TARGET_COLUMNS)

Safe predictors:
['donor_postal_code', 'donor_age', 'gender_identity', 'is_alumnus_flag', 'is_parent_flag', 'preferred_address_type', 'last_fiscal_year_donation', 'donation_2_fiscal_years_ago', 'donation_3_fiscal_years_ago', 'donation_4_fiscal_years_ago', 'donation_5_fiscal_years_ago']

Potential leakage columns:
['has_involvement_flag', 'has_email_flag', 'consecutive_donor_years', 'donor_indicator_flag']

Identifier columns:
['donor_unique_id']

Excluded features:
['is_member_flag', 'current_fiscal_year_donation', 'cumulative_donation_amount']

Target-related columns:
['target_current_fiscal_year_donor_flag']


## Feature Leakage Assessment

A formal leakage audit was completed for all 20 columns currently in the working dataset, including the newly created target variable.

Each column was assigned one of five classifications:

- Safe predictor: Available before the prediction period and does not directly reveal the target
- Potential leakage: May be usable, but its calculation timing or snapshot date must be verified
- Identifier: Used only for record tracking and not as a predictor
- Target: The binary outcome the model will predict
- Excluded feature: Must not be included in the predictor set because it is uninformative or directly risks leakage

The audit identified 11 safe predictors, consisting primarily of demographic, contact-preference, geographic, and completed historical donation variables.

The following four columns were classified as potential leakage:

- `has_involvement_flag`
- `has_email_flag`
- `consecutive_donor_years`
- `donor_indicator_flag`

`has_involvement_flag` and `has_email_flag` may be useful predictors, but their snapshot dates are unknown. They should remain excluded from the initial modeling feature set until it is confirmed that their values were available before the prediction cutoff.

`consecutive_donor_years` may include donation behavior from the current fiscal year. It must either be verified as a historical-only measure or replaced with a streak feature reconstructed from the five completed historical donation years.

`donor_indicator_flag` remains a historical donor-status field rather than the modeling target. However, its calculation logic and reference date are not currently known, so it will not be used as a predictor unless its timing is confirmed.

The following three columns were classified as excluded features:

- `is_member_flag`
- `current_fiscal_year_donation`
- `cumulative_donation_amount`

`is_member_flag` is excluded because it is constant throughout the finalized dataset and provides no predictive variation.

`current_fiscal_year_donation` directly determines the binary target and would cause target leakage if included as a predictor. It will be retained temporarily only for target construction and validation.

`cumulative_donation_amount` may contain current-fiscal-year donations and therefore may include information from the outcome period. The original column will remain excluded. A separate historical-value feature will be created using only donation information available before the prediction cutoff.

`donor_unique_id` was classified as an identifier. It will be retained for tracking records and connecting predictions back to individual donors, but it will not be included in the predictor matrix.

The final modeling target is `target_current_fiscal_year_donor_flag`.

The leakage audit provides the initial rules for predictor eligibility. Columns classified as potential leakage may be reconsidered if reliable documentation confirms that they were available before the prediction baseline.
